# Proxy Essay Generation — vLLM + LLaMA 3.2 3B Instruct

This notebook generates a dataset of **proxy essays** used during development of the adversarial submission pipeline. Rather than calling the competition's judging LLMs directly (which consumes submission quota), a local LLaMA 3.2 3B Instruct model is used via **vLLM** to produce ~1,500 short essays across diverse topics.

These proxy essays serve two purposes:
1. **Behavioural profiling** — understanding how LLM judges score genuinely written essays vs. injected content
2. **Calibration** — verifying that injection payloads do not degrade scores beyond an acceptable threshold when blended with coherent text

The resulting `proxy_essays_llama.xlsx` is consumed by `Solution.ipynb` as reference data.

## Credits

- **@richolson** — vLLM 0.6.3 utility script (Kaggle inference template)
- **@takanashihumbert** — Qwen2.5-32B-AWQ model weights

---

## 1. Setup & Configuration

Imports core libraries, sets the model path to LLaMA 3.2 3B Instruct, and disables tokenizer parallelism to avoid fork-related warnings inside the vLLM subprocess. `IS_SUBMISSION` detects whether the notebook is running as a competition rerun, enabling conditional logic downstream.

In [1]:
import pandas as pd
import numpy as np
import os
import random

### change MODEL PATH here
os.environ['llm_path'] = '/kaggle/input/llama-3.2/transformers/3b-instruct/1'

os.environ["TOKENIZERS_PARALLELISM"] = "false"

IS_SUBMISSION = bool(os.getenv("KAGGLE_IS_COMPETITION_RERUN"))

---

## 2. Load Sample Topics

Loads a preview of the master topic set to verify the CSV structure before running full inference. The dataset contains diverse essay prompts sampled from a larger pool; only the `id` and `topic` columns are used downstream.

In [2]:
df = pd.read_csv('/kaggle/input/sampletopics/MASTER_TOPIC_SET_DATAFRAME_SAMPLED.csv',index_col=0,nrows=5)
df.head(10)

,id,topic
0,2966,The Impact of Technological Advancements on Ur...
1,970,The role of music in promoting social justice ...
2,1386,The Future of Smart Retail
3,1234,The Geopolitics of the Tigris-Euphrates Region
4,2997,The Ethics of Human-AI Collaboration


---

## 3. vLLM Inference Script

Writes `run_vllm.py` to disk using the `%%writefile` magic. Running inference as a subprocess (rather than inline) avoids CUDA context conflicts and makes GPU memory management more predictable across restarts.

**Key design choices:**
- **Model:** LLaMA 3.2 3B Instruct — lightweight enough to run on 2× Kaggle T4s with tensor parallelism (`tensor_parallel_size=2`)
- **Essay length target:** ~70 words — short enough to leave room for injection payloads within any per-essay character limits enforced by the competition
- **Sampling:** `temperature=0.7`, `top_p=0.9` — introduces enough variation across essays to avoid near-duplicate outputs in the proxy set
- **`skip_special_tokens=False`** — preserves special tokens in outputs, useful for inspecting any accidental chat-template leakage during development
- **Output:** Saved to `proxy_essays_llama.xlsx` for direct consumption by `Solution.ipynb`

In [3]:
%%writefile run_vllm.py

import sys
import re
import gc
import vllm
print('vllm version=',vllm.__version__)
import pandas as pd
import os
import random

os.environ["CUDA_VISIBLE_DEVICES"]="0,1"

df = pd.read_csv('/kaggle/input/sampletopics/MASTER_TOPIC_SET_DATAFRAME_SAMPLED.csv',index_col=0)

llm = vllm.LLM(model=os.getenv("llm_path"),
          dtype='half',
          enforce_eager=True,
          gpu_memory_utilization=0.98,
          max_model_len=1024,
          tensor_parallel_size=2,
          trust_remote_code=True)
tokenizer = llm.get_tokenizer()

def apply_template(topic, tokenizer):
    messages = [
        {"role": "system", 
         "content": '''You are an expert essay writer. Write a comprehensive essay on the given topic in English language.
IMPORTANT: Limit the essay to approximately 70 words. Again, please limit essay to less than 70 words, it's important'''
        },
        {
            "role": "user", 
            "content": topic 
        }
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return text

df["topic"] = df['topic'].apply(lambda x: apply_template(x, tokenizer))
print('Example input-\n',df["topic"][0])

seed = random.randint(0, 10000)

responses = llm.generate(
    df["topic"].values,
    vllm.SamplingParams(
        n=1,  # Number of output sequences to return for each prompt.
        top_p=0.9,  # Float that controls the cumulative probability of the top tokens to consider.
        temperature=0.7,  # randomness of the sampling
        seed=seed, # Seed for reprodicibility
        skip_special_tokens=False,  # Whether to skip special tokens in the output.
        max_tokens=199,  # Maximum number of tokens to generate per output sequence.
    ),
    use_tqdm = True
)

df["essay"] = [x.outputs[0].text for x in responses]
df.to_excel("proxy_essays_llama.xlsx", columns=["id", "essay"], index=False)

Writing run_vllm.py


---

## 4. Run Inference

Executes the vLLM script as a subprocess. Generation of ~1,500 essays typically completes in a few minutes on dual T4 GPUs. Progress is streamed via `tqdm`.

In [4]:
!python run_vllm.py

vllm version= 0.6.3.post1
WARNING 01-04 08:13:04 config.py:1668] Casting torch.bfloat16 to torch.float16.
INFO 01-04 08:13:27 config.py:905] Defaulting to use mp for distributed inference
WARNING 01-04 08:13:27 config.py:395] To see benefits of async output processing, enable CUDA graph. Since, enforce-eager is enabled, async output processor cannot be used
INFO 01-04 08:13:27 llm_engine.py:237] Initializing an LLM engine (v0.6.3.post1) with config: model='/kaggle/input/llama-3.2/transformers/3b-instruct/1', speculative_config=None, tokenizer='/kaggle/input/llama-3.2/transformers/3b-instruct/1', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, rope_scaling=None, rope_theta=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.float16, max_seq_len=1024, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=2, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=True, kv_cache_dt

---

## 5. Verify Output

Confirms the generated DataFrame shape and spot-checks essays before saving. The target is 1,500 rows — one essay per topic in the sampled master set.

In [5]:
#df = pd.read_csv('submission.csv')
#df.to_excel("sample_essays.xlsx")

In [6]:
df.shape

(5, 2)